# Ronda 2 · Desarrollo · gemma4:26b-a4b-it-qat

Perfil: `gemma26b`. Se comparan V3 y V4c con los mismos datos, un modelo a la vez. La primera ronda queda intacta.

**No es una validación final ni un experimento aislado del tamaño del modelo.** Se mantienen los parámetros explícitos de la configuración seleccionada y se añade una semilla registrada.

## 1. Verificar archivos, versión e identidad
Lee la configuración compartida y consulta Ollama. No realiza inferencia. No actualiza ni descarga modelos.

In [ ]:
import json
import copy
import hashlib
import re
import sys
import time
import random
from pathlib import Path
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler
from urllib.error import HTTPError, URLError

PERFIL_ID = 'gemma26b'
CONFIG_FILE = "configuracion_ronda2_desarrollo_v1.json"
CONFIG_SHA_ESPERADO = '501db6aa673feab7408738e4a4324a8ac0ade5c50fcdba1666a37162a377614c'

# Se admiten el directorio del notebook, 01_Ronda2, 05_Pruebas o la raíz del repositorio.
# No se busca por todo el disco ni se usa una ruta absoluta de otro equipo.
candidatas = [Path(".."), Path("."), Path("01_Ronda2"), Path("05_Pruebas/01_Ronda2")]
BASE = next((p.resolve() for p in candidatas if (p / CONFIG_FILE).is_file()), None)
if BASE is None:
    raise FileNotFoundError("No se encontró 01_Ronda2. Descomprime el paquete dentro de 05_Pruebas y abre este notebook desde su carpeta.")

bytes_config = (BASE / CONFIG_FILE).read_bytes()
if hashlib.sha256(bytes_config).hexdigest() != CONFIG_SHA_ESPERADO:
    raise ValueError("La configuración cambió. No mezcles esta ejecución con desarrollo_v1; conserva el archivo original y versiona el cambio.")
config = json.loads(bytes_config)
bytes_casos = (BASE / config["casos_file"]).read_bytes()
if hashlib.sha256(bytes_casos).hexdigest() != config["casos_sha256"]:
    raise ValueError("Cambió el archivo de casos. Detén la ejecución y versiona el conjunto antes de continuar.")
casos = json.loads(bytes_casos)["casos"]

# Los archivos de la primera ronda se leen, nunca se modifican.
for nombre, esperado in config["archivos_originales"].items():
    ruta = BASE.parent / nombre
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta el archivo compartido original: {ruta}")
    if hashlib.sha256(ruta.read_bytes()).hexdigest() != esperado:
        raise ValueError(f"No coincide la huella del archivo original {nombre}.")

perfil = next(p for p in config["modelos"] if p["perfil_id"] == PERFIL_ID)
MODELO = perfil["modelo"]
SALIDA = BASE / perfil["carpeta"] / "resultados_ronda2.jsonl"
SALIDA.parent.mkdir(exist_ok=True)
API = "http://127.0.0.1:11434"
if config["api_url"] != API + "/api/chat":
    raise ValueError("La configuración debe utilizar exclusivamente el servidor local.")
cliente = build_opener(ProxyHandler({}))

def leer_api(endpoint, payload=None, timeout=15):
    datos = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    peticion = Request(API + endpoint, data=datos,
                       headers={"Content-Type": "application/json"},
                       method="GET" if payload is None else "POST")
    with cliente.open(peticion, timeout=timeout) as conexion:
        return json.load(conexion)

try:
    version_ollama = leer_api("/api/version")["version"]
    instalados = leer_api("/api/tags")["models"]
except (URLError, TimeoutError) as error:
    raise RuntimeError("No se puede consultar Ollama local. Inicia ollama serve en otra terminal y vuelve a ejecutar esta celda.") from error

if version_ollama != config["ollama_version_esperada"]:
    raise RuntimeError(f"Ollama cambió: {version_ollama}. La ronda fue preparada para {config['ollama_version_esperada']}. Registra y versiona este cambio antes de inferir; no se exige desinstalar ni retroceder de versión.")

por_nombre = {m["name"]: m for m in instalados}
for p in config["modelos"]:
    m = por_nombre.get(p["modelo"])
    if m is None:
        raise RuntimeError(f"No está instalado: {p['modelo']}. No descargues otra etiqueta sin revisar su identidad.")
    if m["digest"] != p["modelo_digest_esperado"]:
        raise RuntimeError(f"Cambió el digest de {p['modelo']}. No continuar con pesos distintos bajo el mismo identificador experimental.")

detalles_modelo = leer_api("/api/show", {"model": MODELO})
print("Ronda:", config["protocolo_id"])
print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Cinco modelos e identidades: verificados")
print("Modelo de esta libreta:", MODELO)
print("Modo solicitado:", perfil["solicitud_base"].get("think"))
print("Opciones heredadas:", perfil["solicitud_base"]["options"])
print("Cambio nuevo para desarrollo: seed =", config["semillas_desarrollo"][0])
print("Casos:", len(casos), "| Paquetes independientes:", len({c["grupo_evidencia"] for c in casos}))
print("V3 original: intacto | V4c: candidato no validado")
print("Salida nueva:", SALIDA)
print("Esta celda no realiza inferencias.")


## 2. Revisar el diseño
Los 12 casos son sintéticos y forman seis parejas. Cambia únicamente la etiqueta dentro de cada pareja. Las guías de revisión viven en `evaluacion_no_enviar` y nunca se incluyen en la solicitud. V4c añade requisitos explícitos de cobertura; no se reinterpretan retroactivamente las omisiones de V3 como alucinaciones.

In [ ]:
# Solo inspección. Las guías de evaluación no se envían al modelo.
for c in casos:
    print(c["case_id"], "|", c["grupo_evidencia"], "|", c["prediccion_recibida"], "|", c["categoria"])

print("\nPILOTO: mismo paquete de evidencia, dos etiquetas distintas.")
for c in casos:
    if c["case_id"] in config["reglas_operativas"]["piloto_case_ids"]:
        print("\n" + c["case_id"] + "\n" + c["contenido"])

print("\nPROMPT V4c CANDIDATO (no sustituye V3):\n")
print(config["prompts"]["V4c"]["system_prompt"])


## 3. Preparar el ejecutor
Cada llamada es una conversación independiente. Se conserva el resultado bruto, incluso errores, truncamientos o razonamiento incrustado. No hay reintentos automáticos ni sustitución de respuestas. Al reejecutar se omite cualquier solicitud ya registrada, exitosa o no. Un fallo se revisa y versiona antes de repetir.

In [ ]:
def hash_objeto(obj):
    b = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(b).hexdigest()


def construir_solicitud(caso, version_prompt, semilla):
    """Lista blanca: no se envían respuestas esperadas ni criterios de evaluación."""
    prompt = config["prompts"][version_prompt]
    solicitud = copy.deepcopy(perfil["solicitud_base"])
    solicitud["stream"] = False
    solicitud["options"]["seed"] = semilla
    solicitud["messages"] = [
        {"role": "system", "content": prompt["system_prompt"]},
        {"role": "user", "content": prompt["user_prompt_template"].format(
            case_id=caso["case_id"], contenido=caso["contenido"])}
    ]
    return solicitud


def analizar_salida(respuesta):
    """Clasifica ejecución y formato; nunca califica automáticamente la calidad."""
    mensaje = respuesta.get("message") or {}
    contenido = mensaje.get("content") or ""
    separado = bool((mensaje.get("thinking") or "").strip())
    incrustado = bool(re.search(r"</?think\b", contenido, re.I))
    final = contenido.strip()
    formato = "sin_think_incrustado"
    if incrustado:
        # Solo separar bloques completos al inicio. Nunca borrar un tramo arbitrario de una evidencia.
        while True:
            bloque = re.match(r"^\s*<think\b[^>]*>.*?</think\s*>\s*", final, re.I | re.S)
            if not bloque:
                break
            final = final[bloque.end():].strip()
        if re.search(r"</?think\b", final, re.I):
            formato = "think_incompleto_o_no_inicial"
            final = ""
        else:
            formato = "think_incrustado_separado"

    razon = respuesta.get("done_reason")
    if razon == "length":
        estado = "truncada"
    elif respuesta.get("done") and razon == "stop" and final:
        estado = "completa"
    else:
        estado = "revisar_ejecucion"

    return {"estado_ejecucion": estado, "respuesta_final_evaluable": final,
            "formato_salida": formato, "razonamiento_separado": separado,
            "razonamiento_incrustado": incrustado,
            "revision_modo": (separado or incrustado) != bool(perfil["solicitud_base"].get("think")),
            "evaluacion_calidad": "pendiente"}


def leer_registros():
    if not SALIDA.exists():
        return []
    filas=[]
    with SALIDA.open(encoding="utf-8") as f:
        for numero, linea in enumerate(f, 1):
            if linea.strip():
                try:
                    filas.append(json.loads(linea))
                except json.JSONDecodeError as error:
                    raise ValueError(f"JSON incompleto en {SALIDA.name}, línea {numero}. Conserva copia del archivo antes de corregirlo.") from error
    return filas


def ejecutar_desarrollo(etapa="piloto"):
    """Una solicitud a la vez, sin reintentos automáticos ni cambios del modelo."""
    if etapa not in ("piloto", "desarrollo"):
        raise ValueError("Usa etapa='piloto' o etapa='desarrollo'.")
    previos = leer_registros()
    # No mezclar silenciosamente una revisión de casos/prompts con registros de este paquete.
    for r in previos:
        if r.get("sha256_config_ronda2") != CONFIG_SHA_ESPERADO:
            raise ValueError("La salida contiene una configuración diferente. Usa una nueva versión o una carpeta nueva.")

    cargados = leer_api("/api/ps").get("models", [])
    ajenos = [m.get("name", m.get("model")) for m in cargados
              if m.get("name", m.get("model")) != MODELO]
    if ajenos:
        raise RuntimeError("Hay otros modelos cargados. Libéralos antes de medir: " + "; ".join("ollama stop " + n for n in ajenos))
    actual = next((m for m in leer_api("/api/tags")["models"] if m["name"] == MODELO), None)
    if actual is None or actual["digest"] != perfil["modelo_digest_esperado"]:
        raise RuntimeError("Cambió la identidad del modelo desde la verificación.")

    seleccionados = [c for c in casos if etapa == "desarrollo" or c["case_id"] in config["reglas_operativas"]["piloto_case_ids"]]
    trabajos=[]
    for semilla in config["semillas_desarrollo"]:
        for c in seleccionados:
            for pv in ("V3", "V4c"):
                req = construir_solicitud(c, pv, semilla)
                llave = hash_objeto({"perfil": PERFIL_ID, "digest":perfil["modelo_digest_esperado"],
                                      "config": CONFIG_SHA_ESPERADO, "solicitud": req})
                trabajos.append((c, pv, semilla, req, llave))
    # Orden reproducible, sin usar una conversación acumulada.
    random.Random(config["semillas_desarrollo"][0]).shuffle(trabajos)
    existentes={r["llave_ensayo"] for r in previos}
    pendientes=[t for t in trabajos if t[4] not in existentes]
    print("Plan:", etapa, "| solicitudes:", len(trabajos), "| ya registradas:", len(trabajos)-len(pendientes), "| pendientes:", len(pendientes))
    if not pendientes:
        print("No hay solicitudes nuevas. No se repitieron inferencias.")
        return

    for c, pv, semilla, req, llave in pendientes:
        registro={
          "execution_id":str(uuid4()),"llave_ensayo":llave,"protocolo_id":config["protocolo_id"],
          "grupo":"desarrollo_ronda2","case_id":c["case_id"],"grupo_evidencia":c["grupo_evidencia"],
          "tipo":"sintetico","prediccion_recibida":c["prediccion_recibida"],
          "perfil_id":PERFIL_ID,"configuracion_modelo":PERFIL_ID+"_r2_desarrollo_v1",
          "modelo_digest":perfil["modelo_digest_esperado"],"prompt_version":pv,"seed":semilla,
          "ollama_version":version_ollama,"python_version":sys.version.split()[0],
          "sha256_config_ronda2":CONFIG_SHA_ESPERADO,"sha256_casos":config["casos_sha256"],
          "sha256_solicitud":hash_objeto(req),"solicitud":req,
          "detalles_modelo":detalles_modelo,"fecha_inicio_utc":datetime.now(timezone.utc).isoformat(),
          "evaluacion_calidad":"pendiente","reintentos":0}
        print("Ejecutando:", c["case_id"], "|", pv, "| seed", semilla, flush=True)
        inicio=time.perf_counter()
        fallo=False
        try:
            respuesta=leer_api("/api/chat", req, timeout=config["reglas_operativas"]["timeout_s"])
            registro["respuesta_original"]=respuesta
            registro.update(analizar_salida(respuesta))
            usados=(respuesta.get("prompt_eval_count") or 0)+(respuesta.get("eval_count") or 0)
            registro["posible_limite_contexto"]=usados >= req["options"]["num_ctx"]
        except KeyboardInterrupt:
            fallo=True
            registro.update(estado_ejecucion="interrumpida", error_tipo="KeyboardInterrupt",
                            error="Interrupción del cliente; no se repite automáticamente. Verificar servidor antes de reintentar.")
        except Exception as error:
            fallo=True
            registro.update(estado_ejecucion="error",error_tipo=type(error).__name__,error=str(error))
            if isinstance(error, HTTPError):
                registro["error_http_cuerpo"]=error.read().decode("utf-8", errors="replace")[:4000]
        registro["tiempo_cliente_s"]=round(time.perf_counter()-inicio,3)
        registro["fecha_fin_utc"]=datetime.now(timezone.utc).isoformat()
        # API/ps es observación posterior, fuera del cronómetro de inferencia.
        if not fallo:
            try:
                registro["modelos_cargados_al_final"]=leer_api("/api/ps")
            except Exception as error:
                registro["aviso_medicion_recursos"]=str(error)
        with SALIDA.open("a",encoding="utf-8") as f:
            f.write(json.dumps(registro,ensure_ascii=False)+"\n")
            f.flush()
        r=registro.get("respuesta_original",{})
        print(c["case_id"], "|", pv, "|", registro["estado_ejecucion"],
              "|", r.get("eval_count","-"), "tokens |", registro["tiempo_cliente_s"], "s", flush=True)
        if etapa=="piloto" and registro.get("respuesta_final_evaluable"):
            print("\nRESPUESTA FINAL (sin mostrar razonamiento):\n"+registro["respuesta_final_evaluable"]+"\n")
        detener=fallo or registro["estado_ejecucion"]!="completa" or registro.get("revision_modo") or registro.get("posible_limite_contexto") or registro.get("razonamiento_incrustado")
        if detener:
            print("Prueba conservada. Se detiene el lote para revisar ejecución, modo o formato; no se reintenta automáticamente.")
            break
    print("Archivo actualizado:", SALIDA)
    print("Completa describe la ejecución, NO una evaluación de calidad aprobada.")


## 4. Piloto deliberado
La primera ejecución deja `EJECUTAR=False`. Para iniciar las cuatro solicitudes del piloto, cambia solo ese valor a `True`. El archivo nuevo `resultados_ronda2.jsonl` queda en esta carpeta. No debe moverse ni fusionarse con `resultados.jsonl` de la primera ronda.

Una vez revisado el piloto, `ETAPA='desarrollo'` completa los 12 casos con ambas instrucciones: 24 solicitudes por modelo, reutilizando las cuatro ya registradas sin repetirlas.

In [ ]:
# Empieza con el piloto: 2 casos (una pareja) × 2 prompts × 1 semilla = 4 solicitudes.
# No cambies a 'desarrollo' antes de revisar entradas, reglas y piloto.
ETAPA = "piloto"
EJECUTAR = False

if EJECUTAR:
    ejecutar_desarrollo(ETAPA)
else:
    print("Verificación preparada. Cambia EJECUTAR a True y ejecuta esta celda para iniciar SOLO el piloto de este modelo.")


## 5. Lectura del resultado
Evalúa `respuesta_final_evaluable`, no el texto de razonamiento. `estado_ejecucion='completa'` solo indica que llegó una respuesta no truncada. El consumo de tokens del servidor puede incluir razonamiento. Las duraciones, carga y caché se conservan; no se mezclan con el tiempo de carga inicial al comparar generación.

La validación reservada no forma parte de este paquete. Antes de realizarla habrá que congelar instrucciones, criterios y datos, conservar etiquetas simuladas como tales y comparar todas las repeticiones, no la mejor respuesta.